In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # 国内镜像，避免下载中断
print("已启用:", os.environ["HF_ENDPOINT"])

已启用: https://hf-mirror.com


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tiny_name = "sshleifer/tiny-gpt2"        # 只有几MB，专供测试
tiny_model = AutoModelForCausalLM.from_pretrained(tiny_name)
tiny_tokenizer = AutoTokenizer.from_pretrained(tiny_name)

print("参数量:", f"{sum(p.numel() for p in tiny_model.parameters()):,}")
print("点火测试通过 ✅")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

E:\Programme\anaconda\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wuziqi\.cache\huggingface\hub\models--sshleifer--tiny-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

参数量: 102,714
点火测试通过 ✅


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"        # 若下载太慢，可改成 "distilgpt2"（约350MB，效果接近）
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

n = sum(p.numel() for p in model.parameters())
print(f"参数量: {n:,}（约 {n/1e8:.2f} 亿）")
print(f"运行设备: {device}")
cfg = model.config
print(f"层数: {cfg.n_layer},  头数: {cfg.n_head},  维度: {cfg.n_embd},  上下文长度: {cfg.n_positions}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

E:\Programme\anaconda\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wuziqi\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

参数量: 124,439,808（约 1.24 亿）
运行设备: cuda
层数: 12,  头数: 12,  维度: 768,  上下文长度: 1024


In [4]:
for s in ["the sun is bright", "unbelievable", "Hello, world!"]:
    toks = tokenizer.tokenize(s)
    print(f"{s!r} → {toks}   ({len(toks)} 个 token)")

'the sun is bright' → ['the', 'Ġsun', 'Ġis', 'Ġbright']   (4 个 token)
'unbelievable' → ['un', 'bel', 'iev', 'able']   (4 个 token)
'Hello, world!' → ['Hello', ',', 'Ġworld', '!']   (4 个 token)


In [5]:
prompt = "The sky is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))

The sky is the limit.

"I will go and get her.

The only thing I will do is kill the guy who is trying to kill me."

The man was a black guy.

"You see? I don't


In [6]:
@torch.no_grad()
def my_generate(prompt, max_new=40):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new):
        logits = model(ids).logits[:, -1, :]       # 只取最后一位的预测
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)     # 拼回序列末尾
    return tokenizer.decode(ids[0], skip_special_tokens=True)

print(my_generate("The sky is"))

The sky is blue," says Gov. Hubert King. And he is right. You can see it.

At least Canada has some of the best air-conditioning cordial intentions of any country in
